In [156]:
import urllib
from tqdm import tqdm
import requests
import pandas as pd
import re
import json
import datetime
from github_helper import from_github

## This notebook adds which politician it is, and what their party is


In [157]:
df = pd.read_csv(from_github("/voting-data/df_votes_all_periods.csv"))

print(df["aktørid"].nunique())
df.head(2)

710


,vote_id,vote_typeid,afstemning_id,aktørid,vote_opdateringsdato
0,1481475,3,5973,1886,2021-01-28T21:27:39.627
1,1481476,1,5973,1039,2018-02-16T10:35:23


In [158]:
request_session = requests.Session()

def get_actor_relations(aktør_id, session = request_session):
    # all_relations = []
    base_url = "https://oda.ft.dk/api/AktørAktør"
    params = {
        '$filter': 
            f'fraaktørid eq {aktør_id} and rolleid eq 15'
        ,'$expand': 'TilAktør, FraAktør'
    }
    response = session.get(base_url, params=params)
    # print(response.url)
    if response.status_code != 200: #If the response is not ok print it and continue, no need to break the program.
        print(f"HTTP error for {aktør_id}: ", response.status_code)
        print("Response text:", response.text)
        return None
    else:
        try:
            data = response.json()
        except ValueError:
            print("Error: Response is not valid JSON")
            print("Response text:", response.text)
            return None
    relations = data.get('value')

    return relations

def build_parties_from_relations(relations_for_actor):
    all_relations = []
    for relation in relations_for_actor:
        # kendt_aktør = relation.get('fraaktørid')
        kendt_aktør = relation.get('FraAktør')
        kendt_aktør_navn = kendt_aktør.get('navn')
        kendt_aktør_id = kendt_aktør.get('id')
        anden_aktør = relation.get('TilAktør')
        relation_start = relation.get('startdato')
        # relation_slut = relation.get('slutdato') #Found via startdato
        typeid = anden_aktør.get('typeid')
        if typeid == 4: #Typeid 4 means it's a party
            party_name = anden_aktør.get('navn')
            # actor_start = anden_aktør.get('startdato') # Not needed
            # actor_end = anden_aktør.get('slutdato')
            # info = [kendt_aktør_id, kendt_aktør_navn, party_name, actor_start, actor_end, relation_start, relation_slut]
            info = [kendt_aktør_id, kendt_aktør_navn, party_name, relation_start]
            all_relations.append(info)
    
    relations_df = pd.DataFrame(all_relations, columns= ["aktørid", "aktør", "party", "relation_start"])
    
    return relations_df

def make_party_intervals_from_all_actors(relations_df_for_all):
    where_has_relation_start = (~relations_df_for_all['relation_start'].isnull())
    df_rel = relations_df_for_all[where_has_relation_start].copy()
    # df_rel = relations_df_for_all.copy()
    # Make sure dates are real datetimes
    df_rel['relation_start'] = pd.to_datetime(df_rel['relation_start'])

    # Sort so "next" makes sense (per actor)
    df_rel = df_rel.sort_values(['aktørid', 'relation_start'])

    # Next relation_start per actor
    df_rel['relation_end'] = (
        df_rel
        .groupby('aktørid')['relation_start']
        .shift(-1)
        - pd.Timedelta(days=1)
    )

    # 1) For each aktørid, detect when the party changes vs previous row
    party_change = (
        df_rel['party']
        != df_rel.groupby('aktørid')['party'].shift()
    )

    # 2) Use cumulative sum to create a "block" id of consecutive same-party rows
    df_rel['block'] = party_change.groupby(df_rel['aktørid']).cumsum()

    # 3) Group by aktørid + party + block and aggregate start/end
    collapsed = (
        df_rel
        .groupby(['aktørid', 'aktør', 'party', 'block'], as_index=False)
        .agg(
            relation_start=('relation_start', 'min'),
            relation_end=('relation_end', 'max')
        )
    )
    # collapsed.drop(columns = 'block', inplace = True)
    collapsed = collapsed.sort_values(['aktørid', 'relation_start'])
    return collapsed

#12 #Nicolai Vammen
# aktør_id = 18723 #Vermund
# aktør_id = 77 #Kristian Thulesen Dahl
# aktør_id = 8319 #WHO IS THIS? I guess we will never know

# relations_for_actor = get_actor_relations(aktør_id)
# relations_df = build_parties_from_relations(relations_for_actor=relations_for_actor)
# relations_for_actor
# relations_df
# party_int = make_party_intervals_from_all_actors(relations_df)
# party_int


In [159]:
unique_actors = df['aktørid'].unique()

# all_actors = []
# for aktørid in tqdm(unique_actors):

all_relations_df = pd.DataFrame()
for aktørid in tqdm(unique_actors):
    relations_json = get_actor_relations(aktørid)
    relations_df = build_parties_from_relations(relations_json)
    all_relations_df = pd.concat([all_relations_df, relations_df])

party_intervals = make_party_intervals_from_all_actors(all_relations_df)
party_intervals
party_intervals.to_csv("./actor-data/party_intervals_raw.csv", index= False)

  0%|          | 0/710 [00:00<?, ?it/s]

100%|██████████| 710/710 [01:13<00:00,  9.73it/s]


In [ ]:
# party_intervals = pd.read_csv(from_github("/actor-data/party_intervals_raw.csv"))

In [166]:
#Now we clean it up
party_intervals = pd.read_csv(from_github("/actor-data/party_intervals_raw.csv"))
where_no_end = (party_intervals['relation_end'].isnull())
party_intervals.loc[where_no_end, 'relation_end'] = pd.Timestamp.today().normalize()
party_intervals.drop(columns = "block", inplace = True)


#We normalize the party names
PARTY_RULES = [
    # Big Danish parties
    (r'^Enhedslisten',                              'Enhedslisten'),
    (r'^Socialdemokratiet$',                        'Socialdemokratiet'),
    (r'^Socialistisk Folkeparti$',                  'Socialistisk Folkeparti'),
    (r'^Dansk Folkeparti$',                         'Dansk Folkeparti'),
    (r'^Venstre, Danmarks Liberale Parti$',         'Venstre'),
    (r'^Det Radikale Venstre$',                     'Radikale Venstre'),
    (r'^Radikale Venstre$',                         'Radikale Venstre'),
    (r'^Det Konservative Folkeparti$',              'Det Konservative Folkeparti'),
    (r'^Liberal Alliance$',                         'Liberal Alliance'),
    (r'^Moderaterne$',                              'Moderaterne'),
    (r'^Ny Alliance$',                              'Liberal Alliance'),
    (r'^Alternativet$',                             'Alternativet'),
    (r'^Frie Grønne, Danmarks Nye Venstrefløjsparti$', 'Frie Grønne'),
    (r'^Venstresocialisterne$',                     'Venstresocialisterne'),

    # "Uden for folketingsgrupperne - <navn>"
    (r'^Uden for folketingsgrupperne\b',            'Uden for Folketingsgrupperne'),

    # Danmarksdemokraterne variants (dash vs en dash)
    (r'^Danmarksdemokraterne\b',                    'Danmarksdemokraterne'),

    # Kristendemokraterne / Kristeligt Folkeparti (same party, renamed)
    # If you prefer to keep them separate, split these into two different canonicals.
    (r'^Kristendemokraterne$',                      'Kristendemokraterne'),
    (r'^Kristeligt Folkeparti$',                    'Kristendemokraterne'),

    # Other Danish / Danish-rooted parties
    (r'^Fremskridtspartiet$',                       'Fremskridtspartiet'),
    (r'^Frihed 2000$',                              'Frihed 2000'),
    (r'^Borgernes Parti\b',                         'Borgernes Parti'),
    (r'^Nye Borgerlige$',                           'Nye Borgerlige'),

    # Greenlandic parties
    (r'^Inuit Ataqatigiit$',                        'Inuit Ataqatigiit'),
    (r'^Siumut$',                                   'Siumut'),
    (r'^Nunatta Qitornai$',                         'Nunatta Qitornai'),
    (r'^Naleraq$',                                  'Naleraq'),

    # Faroese parties
    (r'^Sambandsflokkurin$',                        'Sambandsflokkurin'),
    (r'^Javnaðarflokkurin$',                        'Javnaðarflokkurin'),
    (r'^Tjóðveldisflokkurin$',                      'Tjóðveldi'),
    (r'^Tjóðveldi$',                                'Tjóðveldi'),
]

def normalize_party(name: str) -> str:
    """Map raw party string to canonical party name."""
    if pd.isna(name):
        return name

    for pattern, canonical in PARTY_RULES:
        if re.match(pattern, name):
            return canonical

    # If nothing matches, just return original
    return name

party_intervals['party_clean'] = party_intervals['party'].apply(normalize_party)
print(party_intervals['party'].unique())
print(party_intervals['party_clean'].unique())
party_intervals.drop(columns = "party", inplace=True)

#And we save it
party_intervals.to_csv("./actor-data/party_intervals.csv", index= False)

['Enhedslisten - De Rød-Grønne' 'Enhedslisten' 'Socialdemokratiet'
 'Inuit Ataqatigiit' 'Socialistisk Folkeparti' 'Dansk Folkeparti'
 'Venstre, Danmarks Liberale Parti'
 'Uden for folketingsgrupperne - Søren Espersen'
 'Danmarksdemokraterne - Inger Støjberg'
 'Danmarksdemokraterne – Inger Støjberg'
 'Uden for folketingsgrupperne - Liselott Blixt' 'Det Radikale Venstre'
 'Radikale Venstre' 'Moderaterne' 'Fremskridtspartiet' 'Frihed 2000'
 'Det Konservative Folkeparti' 'Liberal Alliance'
 'Uden for folketingsgrupperne - Kristian Thulesen Dahl'
 'Uden for folketingsgrupperne - Marie Krarup'
 'Uden for folketingsgrupperne - Hans Kristian Skibby'
 'Uden for folketingsgrupperne - Simon Emil Ammitzbøll'
 'Uden for folketingsgrupperne - Simon Emil Ammitzbøll-Bille'
 'Uden for folketingsgrupperne - Dennis Flydtkjær'
 'Uden for folketingsgrupperne - Lars Løkke Rasmussen' 'Ny Alliance'
 'Uden for folketingsgrupperne - Karina Adsbøl'
 'Uden for folketingsgrupperne - Bent Bøgsted'
 'Uden for folket

In [ ]:
print(party_intervals["aktørid"].nunique()) #We have removed 15 actors. All good i hope :D

695
